[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lalo/TorchCode/blob/master/templates/00_b06_performance.ipynb)

# ⭐ Bonus 6: Memory & Performance

Correct code can still be too slow or run out of memory. Most PyTorch performance problems come from a few patterns: copying when a view would do (or the reverse), Python loops over tensor elements, hidden GPU synchronizations, and huge intermediate tensors like an `S × S` attention matrix. This notebook shows how to spot each one and what to do about it.

### What you'll learn
1. Views vs copies: which ops share memory, and why it matters
2. In-place operations: memory savings vs autograd errors
3. Measuring speed properly, and why vectorized code wins
4. Hidden synchronizations: `.item()`, printing, data-dependent shapes
5. Peak memory: chunking, `no_grad`, and activation checkpointing

### Exercises
| # | Function | Use it on |
|---|----------|-----------|
| 1 | `is_view_of(a, b)` | #14 KV Cache, #22 Conv2d |
| 2 | `moving_average(x, k)` | any sliding-window statistic |
| 3 | `chunked_attention(Q, K, V, chunk)` | #05 / #09 attention, #25 Flash Attention |
| 4 | `checkpointed_mlp(x, blocks)` | #13 GPT-2 Block stacks |

### Rules
- No Python loops over tensor elements. Looping over chunks (Exercise 3) or blocks (Exercise 4) is fine.

### How to use this notebook
1. **Run the lesson cells** (sections 1, 2, …) from top to bottom. They are demos: change them and re-run.
2. **Do the exercises in order.** Each one has its own `✏️ YOUR IMPLEMENTATION HERE` cell, followed by a debug cell for checking shapes before you submit.
3. **Submit at any time** with the `check("b06_performance")` cell at the bottom. It grades every exercise at once, and each test is named after its exercise, so a partial score shows which ones are left.
4. **Stuck?** `hint("b06_performance")` gives one hint per exercise; `00_b06_performance_solution.ipynb` has the reference code.
5. **This is a bonus notebook.** No problem requires it. Try what you learn on the problems listed in **🔁 Where to use it** at the bottom.

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import torch
from torch import nn

## 1. Views vs copies

A **view** shares memory with the tensor it came from; a **copy** has its own. Compare `x.untyped_storage().data_ptr()` to tell (`data_ptr()` alone also changes for an offset slice).

| Views (no copy) | Copies |
|-----------------|--------|
| basic slicing `x[1:3]`, `x[:, 0]` | advanced indexing `x[[0, 2]]`, boolean masks `x[mask]` |
| `view`, `transpose`, `permute`, `expand`, `unsqueeze`, `unfold`, `detach` | `clone`, arithmetic results, `contiguous()` of a non-contiguous tensor |
| `reshape` **when possible** | `reshape` when the layout requires it |

Two consequences: writing to a view changes the original, and **a small view keeps its whole base tensor alive**. Slicing one row out of a 1 GB tensor still holds the 1 GB until you `.clone()` the row.

In [ ]:
def same_storage(a, b):
    return a.untyped_storage().data_ptr() == b.untyped_storage().data_ptr()

x = torch.arange(12.).view(3, 4)
print(same_storage(x, x[1:]), same_storage(x, x.T), same_storage(x, x[:, None].expand(3, 2, 4)))   # True True True
print(same_storage(x, x[[0, 2]]), same_storage(x, x.clone()), same_storage(x, x.T.reshape(12)))    # False False False
row = x[1]
row[0] = 100.0
print(x[1, 0])                    # 100.: writing to a view changes the original

## 2. In-place operations

In-place ops (`add_`, `mul_`, `x[...] = ...`) avoid allocating a new tensor. But autograd saves some tensors for backward and tracks a **version counter** on each. Modifying a saved tensor in place makes backward fail ("modified by an inplace operation", see B2).
**Rule:** use in-place updates for buffers, optimizer states and preallocated caches, under `torch.no_grad()` (P7); prefer out-of-place ops inside the differentiable forward pass.

In [ ]:
x = torch.randn(3, requires_grad=True)
y = x.exp()                       # exp saves its OUTPUT for backward
print(y._version)                 # 0
y.add_(1)                         # in place: version 1, backward would now fail
print(y._version)

cache = torch.zeros(4, 8)         # preallocated, e.g. a KV cache
with torch.no_grad():
    cache[2] = torch.randn(8)     # write into it: no new allocation

## 3. Measuring speed

- Use **`torch.utils.benchmark.Timer`**: it warms up, repeats, and (on GPU) synchronizes for you. A single `time.perf_counter()` around one call is noisy.
- **Python loops over elements are the #1 slowdown.** Each tensor op has a fixed overhead of microseconds, so a loop of 10,000 tiny ops costs far more than one op over 10,000 elements.

In [ ]:
import torch.utils.benchmark as benchmark
x = torch.randn(10_000)

def loop_sum(x):
    total = torch.tensor(0.0)
    for v in x:
        total += v
    return total

t_loop = benchmark.Timer("loop_sum(x)", globals={"loop_sum": loop_sum, "x": x}).timeit(3)
t_vec = benchmark.Timer("x.sum()", globals={"x": x}).timeit(1000)
print(f"loop: {t_loop.mean * 1e3:.1f} ms   vectorized: {t_vec.mean * 1e6:.1f} µs   speedup ≈ {t_loop.mean / t_vec.mean:,.0f}x")

## 4. Hidden synchronizations

On a GPU, operations are queued and run **asynchronously**. Anything that needs a value **on the CPU** forces Python to wait for the whole queue: `.item()`, `.tolist()`, `print(tensor)`, `if tensor > 0:`, and ops whose output **shape depends on the data** (`nonzero`, boolean-mask indexing, `unique`).
In a training loop, keep the loss as a tensor and call `.item()` only when you log, not every step.

```python
running = torch.zeros((), device=device)
for step, batch in enumerate(loader):
    loss = train_step(batch)
    running += loss.detach()               # stays on the GPU: no sync
    if step % 100 == 0:
        print(running.item() / 100)        # one sync per 100 steps
        running.zero_()
```

## 5. Peak memory

Memory runs out at the **peak**: the moment the most tensors are alive at once.
- **Big intermediates:** attention's scores are `B × H × S × S`. At `S = 32k` that's a billion numbers per head. **Chunking** (process queries in blocks, as in Exercise 3) or Flash Attention (#25) never materializes it.
- **Autograd saves activations** for backward. For evaluation, run under `torch.no_grad()` / `inference_mode()` so nothing is saved.
- **Activation checkpointing** (`torch.utils.checkpoint.checkpoint`) doesn't save a block's internal activations at all, and recomputes them during backward: roughly 30% more compute for much less memory. Large transformers checkpoint every block.

In [ ]:
from torch.utils.checkpoint import checkpoint
block = nn.Sequential(nn.Linear(16, 64), nn.GELU(), nn.Linear(64, 16))
calls = []
block[0].register_forward_hook(lambda *a: calls.append(1))
x = torch.randn(4, 16, requires_grad=True)
checkpoint(block, x, use_reentrant=False).sum().backward()
print("forward runs of the block's first layer:", len(calls))   # 2: once forward, once recomputed in backward

---
# ✏️ Exercises

Implement the four functions below. Each one has a debug cell you can use before you submit.

### Exercise 1: `is_view_of(a, b)`

Do `a` and `b` share the same underlying memory?

- returns: a Python `bool`, `True` if both tensors use the same storage (even if they start at different offsets or have different shapes)

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def is_view_of(a: torch.Tensor, b: torch.Tensor) -> bool:
    pass  # Replace this

In [ ]:
x = torch.arange(12.).view(3, 4)
print(is_view_of(x[1:], x), is_view_of(x.T, x), is_view_of(x[[0]], x), is_view_of(x.clone(), x))
# expect True True False False

### Exercise 2: `moving_average(x, k)`

The mean of every window of `k` consecutive values along the last dim, **without a Python loop**.

- `x`: `(..., N)` float; `k`: window size, `1 <= k <= N`
- returns: `(..., N - k + 1)` with `out[..., i] = x[..., i : i + k].mean()`

*Hint: with `c = x.cumsum(-1)`, the sum of `x[i : i + k]` is `c[i + k - 1] - c[i - 1]`. Prepend a zero to make the first window work. (`x.unfold(-1, k, 1).mean(-1)` also works, at `O(N·k)` cost.)*

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def moving_average(x: torch.Tensor, k: int) -> torch.Tensor:
    pass  # Replace this

In [ ]:
print(moving_average(torch.tensor([1., 2., 3., 4., 5.]), 2))   # expect tensor([1.5, 2.5, 3.5, 4.5])
print(moving_average(torch.tensor([[1., 2., 3.], [4., 4., 7.]]), 3))   # expect [[2.], [5.]]

### Exercise 3: `chunked_attention(Q, K, V, chunk)`

Scaled dot-product attention, `softmax(Q Kᵀ / √d) V`, computed **one block of queries at a time** so the full `S × S` score matrix never exists.

- `Q`, `K`, `V`: `(B, S, d)`; `chunk`: number of queries per block
- returns: `(B, S, d)`, equal to ordinary attention
- the largest score tensor alive at any time must be `(B, chunk, S)`

*Rows of attention are independent, so query blocks can be processed separately. Flash Attention (#25) goes further and chunks the keys too, merging blocks with the online softmax from P9 §7.*

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

import math

def chunked_attention(Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor, chunk: int) -> torch.Tensor:
    pass  # Replace this

In [ ]:
torch.manual_seed(0)
Q, K, V = torch.randn(2, 10, 8), torch.randn(2, 10, 8), torch.randn(2, 10, 8)
out = chunked_attention(Q, K, V, chunk=4)
ref = torch.softmax(Q @ K.transpose(-2, -1) / 8 ** 0.5, dim=-1) @ V
print(out.shape, torch.allclose(out, ref, atol=1e-5))     # expect (2, 10, 8) True

### Exercise 4: `checkpointed_mlp(x, blocks)`

Run a stack of blocks with **activation checkpointing**, so their internal activations are recomputed during backward instead of stored.

- `x`: the input; `blocks`: a list of `nn.Module`s applied in order
- wrap every block in `torch.utils.checkpoint.checkpoint(block, h, use_reentrant=False)`
- returns: the final output, with the same values and gradients as running the blocks normally

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

from torch.utils.checkpoint import checkpoint

def checkpointed_mlp(x: torch.Tensor, blocks: list[nn.Module]) -> torch.Tensor:
    pass  # Replace this

In [ ]:
torch.manual_seed(0)
blocks = [nn.Sequential(nn.Linear(8, 32), nn.GELU(), nn.Linear(32, 8)) for _ in range(3)]
x = torch.randn(4, 8, requires_grad=True)
out = checkpointed_mlp(x, blocks)
ref = x
for b in blocks:
    ref = b(ref)
print(torch.allclose(out, ref))      # expect True

<!-- Generated from scripts/prereqs/content/b06_performance.py by `make prereqs`. Edit that file instead of this notebook; manual edits are overwritten. -->
## ✅ Submit

Run the judge on **all** exercises in this notebook. Each test is labelled with the exercise it checks.

In [ ]:
from torch_judge import check, hint
check("b06_performance")

In [ ]:
# 💡 Stuck? Uncomment for a nudge
# hint("b06_performance")

---
## 🔁 Where to use it

Re-solve these problems using **Memory & Performance**:

| # | Problem | Difficulty | How | Links |
|:---:|---|:---:|---|---|
| #5 | Scaled Dot-Product Attention | 🟡 Medium | Rewrite attention with `chunked_attention` and compare peak tensor sizes | [Open](05_attention.ipynb) · <a href="https://github.com/lalo/TorchCode/blob/master/templates/05_attention.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/lalo/TorchCode/blob/master/templates/05_attention.ipynb" target="_blank">Colab</a> |
| #13 | GPT-2 Block | 🔴 Hard | Checkpoint each GPT-2 block when stacking several with `checkpointed_mlp` | [Open](13_gpt2_block.ipynb) · <a href="https://github.com/lalo/TorchCode/blob/master/templates/13_gpt2_block.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/lalo/TorchCode/blob/master/templates/13_gpt2_block.ipynb" target="_blank">Colab</a> |
| #14 | KV Cache Attention | 🔴 Hard | Preallocate the KV cache and write into it in place, instead of `torch.cat` copying every step | [Open](14_kv_cache.ipynb) · <a href="https://github.com/lalo/TorchCode/blob/master/templates/14_kv_cache.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/lalo/TorchCode/blob/master/templates/14_kv_cache.ipynb" target="_blank">Colab</a> |
| #22 | Conv2d | 🟡 Medium | `unfold` returns a view: check it with `is_view_of`, then see which step copies | [Open](22_conv2d.ipynb) · <a href="https://github.com/lalo/TorchCode/blob/master/templates/22_conv2d.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/lalo/TorchCode/blob/master/templates/22_conv2d.ipynb" target="_blank">Colab</a> |
| #25 | Flash Attention | 🔴 Hard | Chunked attention is the first half of Flash Attention: add key blocks with online softmax | [Open](25_flash_attention.ipynb) · <a href="https://github.com/lalo/TorchCode/blob/master/templates/25_flash_attention.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/lalo/TorchCode/blob/master/templates/25_flash_attention.ipynb" target="_blank">Colab</a> |

🗺️ See the full recommended order in <a href="https://github.com/lalo/TorchCode/blob/master/LEARNING_PATH.md" target="_blank">LEARNING_PATH.md</a>.